In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import numpy as np
import io

# ==========================================
# 1. Configuration & Hyperparameters
# ==========================================
BASE_DIR = r"Layer 1\Deepfake_Image_Dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "Train")
VAL_DIR =   os.path.join(BASE_DIR, "Validation")
TEST_DIR =  os.path.join(BASE_DIR, "Test")

MODEL_SAVE_PATH = "models"
BEST_MODEL_PATH = os.path.join(MODEL_SAVE_PATH, 'effnet_b0_best_v1.pth')
LATEST_MODEL_PATH = os.path.join(MODEL_SAVE_PATH, 'effnet_b0_latest_v1.pth')

BATCH_SIZE = 32
LEARNING_RATE = 1e-4  # Lower LR is better for transfer learning
EPOCHS = 20
IMG_SIZE = (224, 224) # EfficientNet-B0 default size

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. Augmentation Pipeline
# ==========================================
class JPEGCompression(object):
    def __init__(self, quality_min=40, quality_max=80):
        self.quality_min = quality_min
        self.quality_max = quality_max

    def __call__(self, img):
        if not isinstance(img, Image.Image): return img
        quality = np.random.randint(self.quality_min, self.quality_max)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer)

def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([JPEGCompression()], p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize(IMG_SIZE),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# ==========================================
# 3. Model Definition (EfficientNet-B0)
# ==========================================
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super(DeepfakeDetector, self).__init__()
        # Load pre-trained EfficientNet-B0
        self.network = models.efficientnet_b0(weights='IMAGENET1K_V1')
        
        # Replace the classifier (final layer)
        # EfficientNet-B0 classifier is: (0): Dropout, (1): Linear
        in_features = self.network.classifier[1].in_features
        self.network.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, 1) # Single output for Binary Class
        )

    def forward(self, x):
        return self.network(x)

# ==========================================
# 4. Data Loading Engine
# ==========================================
def get_dataloaders():
    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=get_transforms(train=True))
    val_ds = datasets.ImageFolder(VAL_DIR, transform=get_transforms(train=False))
    
    # Check for Test folder
    test_path = TEST_DIR if os.path.exists(TEST_DIR) else VAL_DIR
    test_ds = datasets.ImageFolder(test_path, transform=get_transforms(train=False))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    return train_loader, val_loader, test_loader

# ==========================================
# 5. Training Engine
# ==========================================
def train_model():
    if not os.path.exists(MODEL_SAVE_PATH): os.makedirs(MODEL_SAVE_PATH)

    model = DeepfakeDetector().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    train_loader, val_loader, _ = get_dataloaders()
    
    best_val_loss = float('inf')
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(EPOCHS):
        # Training Phase
        model.train()
        train_loss, train_correct, total = 0, 0, 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for imgs, labels in loop:
            imgs, labels = imgs.to(device), labels.float().to(device).view(-1, 1)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_correct += (preds == labels).sum().item()
            total += labels.size(0)
            loop.set_postfix(loss=loss.item())

        # Validation Phase
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc="[Val]"):
                imgs, labels = imgs.to(device), labels.float().to(device).view(-1, 1)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (torch.sigmoid(outputs) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        # Metrics
        t_loss = train_loss / len(train_loader)
        t_acc = train_correct / total
        v_loss = val_loss / len(val_loader)
        v_acc = val_correct / val_total
        
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        print(f"Epoch {epoch+1}: Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f} | Val Loss: {v_loss:.4f}")

        scheduler.step(v_loss)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print("🌟 Best Model Saved!")

        torch.save(model.state_dict(), LATEST_MODEL_PATH)

    return history

if __name__ == '__main__':
    train_model()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from tqdm import tqdm
import numpy as np

# ==========================================
# 1. Configuration
# ==========================================
BASE_DIR = r"Layer 1\Deepfake_Image_Dataset"
TEST_DIR = os.path.join(BASE_DIR, "Test")
VAL_DIR =  os.path.join(BASE_DIR, "Validation") # Fallback if Test doesn't exist

MODEL_SAVE_PATH = "models"
BEST_MODEL_PATH = os.path.join(MODEL_SAVE_PATH, 'effnet_b0_best_v1.pth')

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. Model & Transforms
# ==========================================
def get_test_transforms():
    return transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super(DeepfakeDetector, self).__init__()
        self.network = models.efficientnet_b0(weights=None) # Weights loaded from file
        in_features = self.network.classifier[1].in_features
        self.network.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 1)
        )
    def forward(self, x): return self.network(x)

# ==========================================
# 3. Evaluation Function
# ==========================================
def evaluate_only():
    # 1. Check for Test Directory
    target_dir = TEST_DIR if os.path.exists(TEST_DIR) else VAL_DIR
    print(f"[INFO] Evaluating on: {target_dir}")

    test_ds = datasets.ImageFolder(target_dir, transform=get_test_transforms())
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # 2. Load Model
    model = DeepfakeDetector().to(device)
    if not os.path.exists(BEST_MODEL_PATH):
        print(f"ERROR: Model file not found at {BEST_MODEL_PATH}")
        return

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    model.eval()

    # 3. Inference
    probs, preds, targets = [], [], []
    
    print("[INFO] Running Inference...")
    with torch.no_grad():
        for imgs, labels in tqdm(test_loader):
            imgs = imgs.to(device)
            out = model(imgs)
            p = torch.sigmoid(out).cpu().numpy()
            
            probs.extend(p)
            preds.extend((p > 0.5).astype(int))
            targets.extend(labels.numpy())

    # 4. Metrics Output
    auc = roc_auc_score(targets, probs)
    print("\n" + "="*40)
    print(f"ROC-AUC SCORE: {auc:.4f}")
    print("="*40)
    print("\nClassification Report:\n")
    print(classification_report(targets, preds, target_names=test_ds.classes))
    
    # 5. Confusion Matrix Plot
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(targets, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=test_ds.classes, yticklabels=test_ds.classes)
    plt.title('Final Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# ==========================================
# 4. Manual Graph Plotter 
# ==========================================
# Note: Since history is lost when you stop the script, you would usually 
# pass the 'hist' variable here. If you want to plot old data, you'd need 
# to have saved your history to a CSV or JSON.
def plot_manual_graphs(history):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    plt.title('Loss Over Epochs')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'g-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'orange', label='Val Acc')
    plt.title('Accuracy Over Epochs')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

if __name__ == '__main__':
    evaluate_only()